# NanoChat-X — Google Colab Training (local-only checkpoints)

This notebook keeps the model architecture at `block_size=256`, `n_layer=8`, `n_head=8`, and `n_embd=512`, with `batch_size=4`.

**Checkpoint policy:** all checkpoint files and timestamped recovery snapshots stay under `/content/NanoChat-X/out/`. Before training, the notebook validates the main checkpoint and falls back to the newest valid local snapshot if necessary. During training, it copies only stable, loadable checkpoints into timestamped local snapshots and uses `--resume` when a checkpoint exists.

**Important Colab limitation:** `/content` is temporary. Local-only checkpoints can recover from a training-process interruption while the same Colab runtime and filesystem remain available. They cannot survive a Colab runtime reset, disconnection that destroys the VM, or a fresh runtime. Surviving those events requires persistent storage (Drive or another remote store).

In Colab, select **Runtime → Change runtime type → GPU**. Run cells from top to bottom.


In [ ]:
# Configure CUDA allocator before importing torch (best after a fresh runtime restart)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
gc.collect()

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory free: {free / 1024**3:.2f} GiB / {total / 1024**3:.2f} GiB")
    torch.cuda.empty_cache()
else:
    raise RuntimeError("No GPU detected. In Colab, select Runtime → Change runtime type → GPU.")

In [ ]:
# Clone only if the repository is not already present in this runtime.
from pathlib import Path
import subprocess

repo_dir = Path("/content/NanoChat-X")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "https://github.com/mohalkarushikesh/NanoChat-X.git", str(repo_dir)], check=True)
else:
    print("Using existing repository:", repo_dir)
%cd /content/NanoChat-X
!pip install -r requirements.txt


In [ ]:
# Replace src/config.py with a memory-friendlier default configuration.
# Architecture is preserved; batch_size is reduced from 48 to 4.
from pathlib import Path

config_content = '''"""Central configuration for NanoChat-X."""

from __future__ import annotations

from dataclasses import dataclass, field, asdict, fields
from typing import Any


@dataclass
class GPTConfig:
    """Architecture of the causal transformer (a small GPT)."""

    vocab_size: int = 256
    block_size: int = 256
    n_layer: int = 8
    n_head: int = 8
    n_embd: int = 512
    dropout: float = 0.1
    bias: bool = True

    def __post_init__(self) -> None:
        if self.n_embd % self.n_head != 0:
            raise ValueError(
                f"n_embd ({self.n_embd}) must be divisible by n_head ({self.n_head})"
            )


@dataclass
class TrainConfig:
    """Everything about a training run (model config included)."""

    model: GPTConfig = field(default_factory=GPTConfig)
    data_path: str = "data/data.txt"
    tokenizer: str = "word"
    val_fraction: float = 0.1
    batch_size: int = 4
    grad_accum_steps: int = 1
    max_iters: int = 25000
    learning_rate: float = 5e-4
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0
    warmup_iters: int = 300
    lr_decay_iters: int = 25000
    min_lr: float = 3e-5
    eval_interval: int = 250
    eval_iters: int = 50
    log_interval: int = 50
    out_dir: str = "out"
    device: str = "auto"
    seed: int = 1337
    compile: bool = False

    def resolved_device(self) -> str:
        if self.device != "auto":
            return self.device
        import torch
        return "cuda" if torch.cuda.is_available() else "cpu"

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict[str, Any]) -> "TrainConfig":
        d = dict(d)
        model = d.pop("model", {})
        cfg = cls(**d)
        cfg.model = GPTConfig(**model)
        return cfg

    def apply_overrides(self, overrides: dict[str, Any]) -> None:
        """Set fields by name; GPTConfig fields are routed to model."""
        model_fields = {f.name for f in fields(GPTConfig)}
        train_fields = {f.name for f in fields(TrainConfig)}
        for key, value in overrides.items():
            if key in model_fields:
                setattr(self.model, key, _coerce(getattr(self.model, key), value))
            elif key in train_fields and key != "model":
                setattr(self, key, _coerce(getattr(self, key), value))
            else:
                raise KeyError(f"Unknown config field: {key!r}")
        self.model.__post_init__()


def _coerce(current: Any, value: Any) -> Any:
    """Cast CLI strings to the type of the existing default."""
    if isinstance(value, str):
        if isinstance(current, bool):
            return value.lower() in ("1", "true", "yes", "y")
        if isinstance(current, int) and not isinstance(current, bool):
            return int(value)
        if isinstance(current, float):
            return float(value)
    return value
'''

Path('src/config.py').write_text(config_content, encoding='utf-8')
print('Updated src/config.py')
print(Path('src/config.py').read_text()[:700])

In [ ]:
# Preprocess the Cornell dataset
!python -m src.preprocess_cornell

In [ ]:
# Verify the model/training defaults before starting
from src.config import GPTConfig, TrainConfig
cfg = TrainConfig()
print("Model:", cfg.model)
print("Batch size:", cfg.batch_size)
print("Gradient accumulation configured:", cfg.grad_accum_steps)
print("Note: grad_accum_steps is 1 unless src/train.py explicitly implements accumulation.")

In [ ]:
# Train with local-only, validated timestamped checkpoints. No Google Drive is used.
from pathlib import Path
from datetime import datetime
import os
import shutil
import subprocess
import time
import torch

repo_dir = Path("/content/NanoChat-X")
out_dir = repo_dir / "out"
checkpoint = out_dir / "ckpt.pt"
snapshot_dir = out_dir / "recovery_checkpoints"
snapshot_dir.mkdir(parents=True, exist_ok=True)

def checkpoint_is_loadable(path: Path) -> bool:
    """Reject missing, empty, partial, or unreadable checkpoint files."""
    if not path.is_file() or path.stat().st_size == 0:
        return False
    try:
        torch.load(path, map_location="cpu", weights_only=False)
        return True
    except Exception as exc:
        print(f"Checkpoint validation failed for {path.name}: {type(exc).__name__}: {exc}")
        return False

def newest_valid_snapshot():
    candidates = sorted(snapshot_dir.glob("ckpt_*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
    for candidate in candidates:
        if checkpoint_is_loadable(candidate):
            return candidate
    return None

# Recover a corrupt/missing main checkpoint from the newest valid local snapshot.
if checkpoint.exists() and not checkpoint_is_loadable(checkpoint):
    print("Main checkpoint is not loadable; trying local recovery snapshots...")
    checkpoint.unlink(missing_ok=True)
if not checkpoint.exists():
    recovery = newest_valid_snapshot()
    if recovery is not None:
        checkpoint.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(recovery, checkpoint)
        if not checkpoint_is_loadable(checkpoint):
            checkpoint.unlink(missing_ok=True)
            raise RuntimeError("Could not restore a valid checkpoint from the local snapshot.")
        print(f"Recovered checkpoint from {recovery.name}")

# If the current checkpoint is valid, make a baseline snapshot before training.
if checkpoint_is_loadable(checkpoint):
    stamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S_%f")
    baseline = snapshot_dir / f"ckpt_{stamp}_baseline.pt"
    shutil.copy2(checkpoint, baseline)
    print("Existing checkpoint found; training will resume from it.")
    resume_args = ["--resume"]
else:
    print("No valid checkpoint found; starting a new training run.")
    resume_args = []

print("Local recovery snapshots:", snapshot_dir)
cmd = ["python", "-m", "src.train", *resume_args]
print("Command:", " ".join(cmd))
process = subprocess.Popen(cmd, cwd=str(repo_dir))
last_signature = None
snapshot_count = 0

def snapshot_if_valid():
    global last_signature, snapshot_count
    if not checkpoint.is_file():
        return
    try:
        stat = checkpoint.stat()
        signature = (stat.st_size, stat.st_mtime_ns)
        if signature == last_signature or stat.st_size == 0:
            return
        # Wait for the writer to finish, then validate before copying.
        time.sleep(2)
        if not checkpoint.is_file():
            return
        stable_stat = checkpoint.stat()
        stable_signature = (stable_stat.st_size, stable_stat.st_mtime_ns)
        if stable_signature != signature or not checkpoint_is_loadable(checkpoint):
            return
        stamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S_%f")
        temp_path = snapshot_dir / f".tmp_{stamp}.pt"
        final_path = snapshot_dir / f"ckpt_{stamp}.pt"
        shutil.copy2(checkpoint, temp_path)
        # Validate the copied file too; publish it atomically only after validation.
        if checkpoint_is_loadable(temp_path):
            os.replace(temp_path, final_path)
            snapshot_count += 1
            print(f"Saved valid local snapshot: {final_path.name} ({final_path.stat().st_size:,} bytes)")
            last_signature = stable_signature
        else:
            temp_path.unlink(missing_ok=True)
    except FileNotFoundError:
        # Training may be in the middle of replacing the checkpoint. Try again next loop.
        pass

try:
    while process.poll() is None:
        snapshot_if_valid()
        time.sleep(2)
    # One final attempt after the training process exits.
    snapshot_if_valid()
    if process.returncode != 0:
        print(f"Training process exited with code {process.returncode}.")
        print("If a valid checkpoint was saved, rerun this cell to resume from it.")
    else:
        print("Training process finished successfully.")
except KeyboardInterrupt:
    print("Interrupt received. Stopping the training subprocess...")
    process.terminate()
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    # Save only a fully loadable checkpoint; do not replace a good snapshot with a corrupt file.
    snapshot_if_valid()
    print("Stopped. Rerun this cell to resume from the newest valid local checkpoint.")
finally:
    # Never promote an unvalidated checkpoint to a recovery snapshot.
    print(f"Valid snapshots created this run: {snapshot_count}")
    valid = [p for p in snapshot_dir.glob("ckpt_*.pt") if checkpoint_is_loadable(p)]
    print(f"Valid local recovery snapshots available: {len(valid)}")


## Recovery behavior and limitations

- All checkpoints are stored locally in `out/` and `out/recovery_checkpoints/`; this notebook does not mount or write to Google Drive.
- The training process writes its active checkpoint to `out/ckpt.pt` (assuming `src/train.py` uses the configured `out_dir`). The watcher waits for a stable file, verifies it can be loaded, then atomically publishes a timestamped snapshot.
- When rerun, the notebook validates `out/ckpt.pt`; if it is missing or corrupt, it restores the newest valid local snapshot. It adds `--resume` only when a valid checkpoint exists.
- This protects against some interrupted/partial checkpoint writes **only while the same Colab runtime's local disk survives**. If the runtime is factory-reset or replaced, `/content` and every local snapshot are lost. Local-only storage cannot restore weights after that event.
- Exact continuation also depends on `src/train.py` saving and loading optimizer state and iteration/step state in its checkpoint. This notebook cannot add those fields if the training script does not save them.
